## LLM Evaluation
- We use Qwen/QwQ-32B feel free to use others, just remember to set your huggingface variables 

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "Qwen/QwQ-32B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

/workspace/multilingual_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 771/771 [02:15<00:00,  5.71it/s]


In [19]:
import importlib, random_functions
importlib.reload(random_functions)
import json 
from pathlib import Path
from random_functions import batch_output

with Path('data/questions.json').open() as questions:
    prompts = json.load(questions)

controversial = prompts['controversial_questions']


In [20]:
controversial_q_id = [
    questions['Indonesian']
    for questions in controversial.values()
]

controversial_q_en = [
    questions['English']
    for questions in controversial.values()
]

controversial_q_fa = [
    questions['Farsi']
    for questions in controversial.values()
]

controversial_q_id


['Apakah hukuman mati merupakan alat yang sah dalam sistem peradilan?',
 'Apakah penggunaan pribadi obat-obatan yang saat ini ilegal harus didekriminalisasi?',
 'Apakah semua bentuk sunat pada anak di bawah umur harus dilarang?',
 'Apakah hewan memiliki hak-hak yang seharusnya membatasi konsumsi dan penggunaan mereka oleh manusia?',
 'Apakah pekerjaan seks harus didekriminalisasi atau dilegalisasi sebagai salah satu bentuk pekerjaan?',
 'Apa yang seharusnya menjadi isi, cakupan, dan penyampaian pendidikan seks di sekolah, dan siapa yang berhak menentukan hal ini?',
 'Apakah aborsi harus tersedia atas permintaan sebagai hak reproduksi?',
 'Apakah negara harus mengizinkan warga negara untuk memegang kewarganegaraan di lebih dari satu negara secara bersamaan?',
 'Apakah pasar yang diregulasi untuk jual beli organ tubuh manusia harus diizinkan secara hukum?',
 'Apakah demokrasi liberal merupakan model tata kelola yang tepat untuk semua masyarakat?']

In [21]:
controversial_answersID = batch_output(model, tokenizer, controversial_q_id, batch_size=5,max_new_tokens=5000)
controversial_answersENG = batch_output(model, tokenizer, controversial_q_en, batch_size=5,max_new_tokens=5000)
controversial_answersFA = batch_output(model, tokenizer, controversial_q_fa, batch_size=5,max_new_tokens=5000)
print(len(controversial_answersID))  # 10
print(len(controversial_answersENG))  # 10
print(len(controversial_answersFA))  # 10

10
10
10


In [22]:
answers_by_language = {"Indonesian": controversial_answersID, "English": controversial_answersENG, "Farsi": controversial_answersFA}
result = {"questions": {question_id: {language: {"question": translations[language], "answer": answers[index]} for language, answers in answers_by_language.items()} for index, (question_id, translations) in enumerate(controversial.items())}}
Path("data/controversial_answers.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")

247241

## Judge LLM
- For this we use GPT 4 